# SDN on ME4OH full-field OFAM3 SST data

Experiment 4: Using SDN only to produce reconstructions

In [ ]:
# Standard imports
import json
import os

# Third-party imports
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler

# Local imports
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 
from helper_functions.ofam3_data import load_files
from helper_functions.preprocess_SDN import create_sdn_datasets
from helper_functions.preprocess_SHRED import split_ordered_data
from helper_functions.saving import JsonEncoder

np.random.seed(42)

## SHRED preprocessing

In [ ]:
exp_index = 4
test_lags = [2, 52]

# Experiment 4.1:
# num_sensors = 5

# Experiment 4.2:
num_sensors = 1059


In [ ]:
# Load and reformat data
data, dates, lats, longs = load_files()

In [ ]:
print("Preprocessing data...")
train_data, val_data, test_data, test_dates = split_ordered_data(data, dates)

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
train_transformed = sc.transform(train_data)  
val_transformed = sc.transform(val_data)
test_transformed = sc.transform(test_data)

In [ ]:
# Build dataset
sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
sensor_coords=(longs[sensor_locations], lats[sensor_locations])

train_dataset = create_sdn_datasets(train_transformed, sensor_locations)
val_dataset = create_sdn_datasets(val_transformed, sensor_locations)
test_dataset = create_sdn_datasets(test_transformed, sensor_locations)

In [ ]:
# Generate test set to match SHRED test set
# (use final sensor results for what each sequence would be if using SHRED)
test_dataset_match_shred = {}
test_lag_dates = {}
for lags in test_lags:
    test_dataset_match_shred[lags] = create_sdn_datasets(test_transformed, sensor_locations, test_lags=lags)
    test_lag_dates[lags] = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

## Train + test SDN model

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sdn_model = models.SDN(num_sensors, len(lats)).to(device)
validation_errors = models.fit(sdn_model, train_dataset, val_dataset, batch_size=64,
                               num_epochs=1000, lr=1e-3, verbose=True, patience=5)

In [ ]:
test_recons = sc.inverse_transform(sdn_model(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

In [ ]:
test_recons_match_shred = {}
ground_truth_match_shred = {}
for lags, dataset in test_dataset_match_shred.items():
    test_recon_lags = sc.inverse_transform(sdn_model(dataset.X).detach().cpu().numpy())
    ground_truth_lags = sc.inverse_transform(dataset.Y.detach().cpu().numpy())
    print('Test Reconstruction Error: ')
    print(np.linalg.norm(test_recon_lags - ground_truth_lags)/ np.linalg.norm(ground_truth_lags))
    test_recons_match_shred[lags] = test_recon_lags
    ground_truth_match_shred[lags] = ground_truth_lags

## Save results

In [ ]:
# Make experiment folder
directory = f"Reconstructions/Static/Exp{exp_index}/"
os.makedirs(directory, exist_ok = True)

In [ ]:
np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}", test_recons)
np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}", test_ground_truth)

for lags in test_lags:
    np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}", test_recons_match_shred[lags])
    np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}", ground_truth_match_shred[lags])

In [ ]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_dates": list(test_dates)
    }

In [ ]:
with open(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}.json",
          'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)

# Save metadata for matching lags testing
for lags in test_lags:
    metadata["test_lag_dates"] = test_lag_dates[lags]
    with open(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}_l{lags}.json",
          'w', encoding='utf-8') as f:
      json.dump(metadata, f, cls=JsonEncoder)